# 🎬 Piku vs 3 Hyenas — 3D Cartoon Generator (With Hindi Voice!)
### ⚡ Running 100% on Colab Free T4 GPU (Zero Quota Limits • Zero Blur)


## 📦 Step 1: Install High-Speed Diffusers & Fetch Storyboard Assets + Audio

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf imageio[ffmpeg]
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Fresh clone of repository
!rm -rf videogen
!git clone https://github.com/yashkumarsahu789/videogen.git

import os, torch
assets_dir = "videogen/video_output/piku_vs_hyenas_slapstick/assets"
audio_file = "videogen/video_output/piku_vs_hyenas_slapstick/audio/piku_vs_hyenas_voiceover_master.mp3"

assert torch.cuda.is_available(), "❌ GPU not found! Go to Runtime -> Change runtime type -> Select T4 GPU -> Save"
print(f"✅ Connected to Colab GPU: {torch.cuda.get_device_name(0)}")
print("✅ 5 Scene Images:", os.listdir(assets_dir))
print("✅ Master Voiceover:", os.path.exists(audio_file))

## 🚀 Step 2: Load Lightricks LTX-Video on Colab GPU (Runs Locally — No Quota Block!)

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import load_image, export_to_video
from pathlib import Path
import os, time

print("📥 Loading Lightricks/LTX-Video DiT Pipeline directly on your Colab GPU...")
pipe = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)
pipe.to("cuda")
print("✅ Model loaded on T4 GPU! No API limits, no queue!")

assets_path = Path("videogen/video_output/piku_vs_hyenas_slapstick/assets")
output_dir = Path("output_clips")
output_dir.mkdir(parents=True, exist_ok=True)

scenes = [
    ("scene1_hyenas_leaf_slide.jpg", "clip1.mp4", "Three goofy spotted hyenas comically sliding fast on giant green leaves down dirt path, screaming wide eyed, 3D Pixar cartoon"),
    ("scene2_armadillo_bowling.jpg", "clip2.mp4", "Cheeky baby monkey in coconut helmet laughing mischievously as he bowls a round spiky armadillo down the slope"),
    ("scene3_crocodile_crash.jpg", "clip3.mp4", "Sliding hyenas crash full-speed into the soft round belly of a giant chubby crocodile basking with a smug smirk"),
    ("scene4_giant_spider_launch.jpg", "clip4.mp4", "Giant fuzzy purple cartoon spider emerging from tree trunk, scaring hyenas who launch screaming into mid-air"),
    ("scene5_monkey_victory_laugh.jpg", "clip5.mp4", "Baby monkey Piku giving double thumbs up smiling playfully, three dizzy hyenas hanging upside down from ravine log")
]

clip_paths = []
for idx, (img_name, out_name, prompt) in enumerate(scenes):
    img_file = assets_path / img_name
    out_file = output_dir / out_name
    print(f"\n🎬 Rendering Scene {idx+1}/5: {img_name}...")
    t0 = time.time()
    
    image = load_image(str(img_file)).resize((704, 512))
    generator = torch.Generator(device="cuda").manual_seed(42 + idx)
    
    # 25 frames @ 10fps = exactly 2.5s per scene (syncs with audio!)
    video_frames = pipe(
        image=image,
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted, glitch, dark",
        width=704,
        height=512,
        num_frames=25,
        num_inference_steps=20,
        generator=generator
    ).frames[0]
    
    export_to_video(video_frames, str(out_file), fps=10)
    clip_paths.append(str(out_file))
    print(f"✓ Rendered in {round(time.time() - t0, 1)}s -> {out_name}")

print(f"\n🎉 ALL 5 SCENES RENDERED SUCCESSFULLY! Total clips: {len(clip_paths)}")

## 🎶 Step 3: Concat Video Clips & Sync Hindi Cartoon Voiceover

In [ ]:
assert len(clip_paths) == 5, f"Expected 5 clips, but found {len(clip_paths)}!"

with open("concat_list.txt", "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.abspath(cp)}'\n")

audio_path = "videogen/video_output/piku_vs_hyenas_slapstick/audio/piku_vs_hyenas_voiceover_master.mp3"
final_output = "piku_vs_hyenas_3d_master_with_voice.mp4"

!ffmpeg -y -f concat -safe 0 -i concat_list.txt -i {audio_path} -c:v libx264 -c:a aac -shortest -pix_fmt yuv420p {final_output}
print(f"\n✅ Master 3D Cartoon Video with Synchronized Voice Ready: {final_output}")

## 📥 Step 4: Play with Voice & Download

In [ ]:
from IPython.display import HTML, FileLink
from base64 import b64encode

mp4_bytes = open(final_output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

display(HTML(f'''
<div style="background:#1E293B; padding:20px; border-radius:16px; border:2px solid #10B981; max-width:760px; margin:auto; text-align:center;">
    <h3 style="color:#F59E0B; margin-top:0;">🐵 Piku vs Hyenas 3D Cartoon (With Hindi Voice!)</h3>
    <video width=704 height=512 controls autoplay loop style="border-radius:12px; width:100%; border:2px solid #334155;">
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
'''))

print("\n⬇️ Click Below to Download Video:")
display(FileLink(final_output))